# Session 11 — Exception Handling in Python

Welcome to Session 11, where we stop fearing errors and start *managing* them. Every program, no matter how carefully written, can fail. The difference between a beginner and a professional is how gracefully that failure is handled: a crashing script that dumps a cryptic traceback is annoying; a script that catches the problem, tells the user what went wrong and continues is robust.

Errors arrive in two distinct stages:

1. **During compilation (parse time)** — the code violates Python's grammar: a missing colon, an unclosed bracket or wrong indentation. These are **syntax errors**. The program does not even start running; the parser stops at the first mistake.
2. **During execution (runtime)** — the code is grammatically fine but something goes wrong while it runs: dividing by zero, indexing past the end of a list, a file that does not exist, or a misspelled variable name. These are **exceptions**.

The good news is that Python's exception system is one of the friendliest in any language. Every failure is an *object* you can catch, inspect and respond to. The `try`/`except` block lets you surround risky code with a net; `else` and `finally` fine-tune the flow; the `raise` keyword lets you throw errors of your own; and custom exception classes let you express domain-specific problems (like *insufficient balance*) by name.

Here is the roadmap:

1. Syntax errors at compile time
2. Common built-in exceptions
3. `try`/`except` — the basic handling pattern
4. Catching specific exceptions
5. `else` and `finally`
6. Raising exceptions with `raise`
7. Custom exception classes
8. The exception hierarchy — one tree that explains what every `except` catches

You will deliberately break code on purpose in the syntax-error cells — the point of this session is that failing loudly and reading the message is how Python teaches you best. Let us get started.

# 1. Syntax Errors — Failing Before You Run

## 1.1 What Is A Syntax Error?

A syntax error means the program text does not follow the grammar of the Python language. It is detected by the interpreter while the code is being parsed — before a single statement executes.

Key facts to remember:

- The error is raised by the compiler/interpreter during compilation, not by your logic.
- The program never runs; the parser fails immediately.
- It is fully fixable: correct the offending line and the obstacle disappears.
- The message usually points at the exact line and leaves a caret (`^`) under the spot where the parser got confused.

Think of syntax as the road signs of a language. A program with a syntax error is like a sentence with a broken word — a human can sometimes guess the meaning, but Python will not. Grammar mistakes block everything else, so they always have the highest debugging priority.

## Syntax Error — compile-time

```python
print 'hello world'   # missing parentheses!
```

- Happens at compile/interpreter time — the program doesn't run, it fails directly
- The Python interpreter caught a mistake in its grammar
- Fix: rewrite the line with correct grammar
- Syntax errors are debuggable — the error message tells you where


## Deep Dive : Syntax errors — compile time

```python
print 'hello world'     # ❌ SyntaxError — missing ()
a = 5
if a==3                 # ❌ missing :
    print('hello')
iff a==3:               # ❌ typo keyword
```

- **Syntax error** — at parse time; Python can't compile the code
- The program doesn't even RUN — the parse fails first
- Causes: missing colon/parens, typo keywords, bad indentation
- IndentationError — the block was set up wrong
- Fix: correct syntax per Python rules
- Highest priority — fix the syntax first
- Debug: the error line + caret pointer
- vs runtime exception — an error during execution
- Syntax is not a problem with the code's logic, just the language rules


In [3]:
# Examples of syntax error
print 'hello world'

SyntaxError: ignored

## 1.2 Typical Sources of Syntax Errors

Most syntax errors come from a small set of careless mistakes:

- **Leftover or missing symbols** — forgetting a colon, or forgetting to close a bracket, parenthesis or quote.
- **Misspelled keywords** — typing `iff` instead of `if`, or `whiler` instead of `while`.
- **Incorrect indentation** — mixing tabs and spaces, or indenting a block inconsistently; Python uses indentation to mark blocks, so it is picky.
- **Empty blocks** — an `if`, `else`, loop, class or function whose body is completely empty (Python expects at least one statement, usually `pass`).

The next few code cells show these failures in action. Run each cell, read the error line, and fix the code to make it run.

## 1.4 Indentation Shapes The Block

Indentation is not a style preference in Python — it is the block structure. After any header line ending in `:`, the statements that belong to it must be indented consistently (usually four spaces).

- `if a == 3:` then a plain `print('hello')` at column zero is `IndentationError`.
- `print('hello')` indented by two spaces, and another line indented by four, makes Python raise `TabError` or a `SyntaxError` about inconsistent use of tabs and spaces.
- Choose spaces over tabs, and keep the same indent width inside every block.

The colon-plus-indentation pair is the single most common cause of syntax errors for beginners. Once you train your eye to look for a missing `:` and a consistent indent, you will fix these in seconds.

## 1.3 Debugging The Three Classics

The next three code cells are ready-made crashes. They are meant to be read, not feared:

- **Missing colon** — `if a==3` with no trailing `:` raises `SyntaxError: invalid syntax`. The colon announces that a suite (an indented block) follows.
- **Misspelled keyword** — `iff a==3:` is not `if`, so the parser rejects it as an unknown name at that position. Python keywords are an exact vocabulary: `if`, `elif`, `else`, `while`, `for`, `def`, `return` and so on.
- **Broken indentation** — `if a==3:` followed by a `print()` that is not indented triggers `IndentationError: expected an indented block`. Because Python has no braces, indentation *is* the grouping mechanism; a body must be indented one level deeper than the header.

When you see `SyntaxError` or `IndentationError`, your first instinct should be: what did I type wrong, and where does the caret point? The error position identifies the line; your eye identifies the fix.

In [68]:
a = 5
if a==3
  print('hello')

SyntaxError: ignored

In [69]:
a = 5
iff a==3:
  print('hello')

SyntaxError: ignored

In [70]:
a = 5
if a==3:
print('hello')

IndentationError: ignored

# 2. Exceptions — Runtime Failures

## 2.1 The Common Built-in Exceptions

A running program can fail for reasons that have nothing to do with grammar. These runtime failures are **exceptions**. Python defines a whole family of built-in ones, each with a name that describes its cause:

| Exception | When it happens |
|---|---|
| `IndexError` | Accessing an invalid index, e.g. `L[100]` on a 3-element list |
| `ModuleNotFoundError` | `import` of a module that cannot be found |
| `KeyError` | A key that is not in the dictionary, e.g. `d['age']` |
| `TypeError` | An operation applied to an inappropriate type, e.g. `1 + 'a'` |
| `ValueError` | A function receives a value it cannot use, e.g. `int('a')` |
| `NameError` | A name that has never been defined, e.g. `print(k)` |
| `AttributeError` | Accessing an attribute/method the object does not have, e.g. `L.upper()` |

Every exception stops normal execution and produces a **stack trace** — a report showing where the error occurred. The first diagnostic skill you should build is simply *reading* that message and connecting the exception name to its cause.

## 2.2 Reading The Traceback

When an exception escapes unhandled, Jupyter shows a traceback whose final line is the actual error (`IndexError: list index out of range`, `TypeError: unsupported operand ...`). Read it bottom-up: the last line names the exception and a human-readable reason; the lines above are the call chain that led to it.

A few rules of thumb:

- The exception **name** is a strong hint: `IndexError` is about indices, `KeyError` about dict keys, `TypeError` about types, `ValueError` about values.
- `ZeroDivisionError` appears whenever you divide by zero; `FileNotFoundError` whenever `open()` cannot find a path.
- Exceptions form a hierarchy rooted at `BaseException`, with `Exception` as the common ancestor of nearly everything you will catch.
- The traceback's bottom line is the truth; the top lines are context.

You are not expected to memorise the list — you are expected to recognise the pattern. Run the next eight cells one by one and let each crash teach you its exception.

In [71]:
# IndexError
# The IndexError is thrown when trying to access an item at an invalid index.
L = [1,2,3]
L[100]

IndexError: ignored

In [73]:
# ModuleNotFoundError
# The ModuleNotFoundError is thrown when a module could not be found.
import mathi
math.floor(5.3)

ModuleNotFoundError: ignored

In [74]:
# KeyError
# The KeyError is thrown when a key is not found

d = {'name':'nitish'}
d['age']

KeyError: ignored

In [78]:
# TypeError
# The TypeError is thrown when an operation or function is applied to an object of an inappropriate type.
1 + 'a'

TypeError: ignored

In [76]:
# ValueError
# The ValueError is thrown when a function's argument is of an inappropriate type.
int('a')

ValueError: ignored

In [79]:
# NameError
# The NameError is thrown when an object could not be found.
print(k)

NameError: ignored

In [80]:
# AttributeError
L = [1,2,3]
L.upper()

# Stacktrace

AttributeError: ignored

# 3. Handling Exceptions — try / except

## 3.1 Why Handling Matters

Exceptions are raised by the Python runtime whenever something unforeseen happens during execution: memory pressure, division by zero (a logical error), a missing resource, a failed network call, a database error. Without handling, such an exception crashes the whole program and dumps an ugly error in the user's face.

Handling flips that: you place the risky operation inside a `try` block, and the fallback behaviour inside an `except` block. If nothing goes wrong, `except` is skipped; if an exception appears, the `try` block is abandoned at that point and the matching `except` handler takes over.

```python
try:
    with open('sample1.txt', 'r') as f:
        print(f.read())
except:
    print('sorry file not found')
```

The user sees a friendly message and the program keeps running instead of dying. That is the entire motivation for this session: handle failures *where they can happen*, and let the user continue.

## 3.2 The try / except Pattern

`try`/`except` is Python's safety net. The mechanics are simple:

- The `try` block holds the code that *may* raise an exception.
- The `except` block holds the fallback that runs *if* an exception occurs.
- If the `try` block completes without errors, the `except` block is ignored entirely.
- If an exception occurs partway through `try`, control jumps immediately to `except` and the rest of the `try` body never runs.

A bare `except:` catches literally anything — convenient for demos, but a clumsy production habit because it swallows the error without telling you which exception it was. The common files are handled with *specific* `except` clauses (next topic). A warning: even `except Exception as e:` is a broad net; the idiomatic path is to catch the specific exceptions your code can actually react to.

## 3.3 What Happens Inside try

A note on how the net behaves:

- Once an exception is raised, the remainder of the `try` block is skipped — statements after the failing line never execute.
- The matching handler is the first `except` clause whose exception class matches, checked top to bottom.
- The code demo above opens `sample1.txt` for reading; that file does not exist, `open()` raises `FileNotFoundError`, the friendly message prints, and the notebook keeps running — no crash, no red traceback.

The same pattern is used everywhere real programs live: opening files, validating input, calling APIs, parsing data. Wrap risk, react gracefully, and let the rest of the program continue.

In [ ]:
# Why is it important to handle exceptions
# how to handle exceptions
# -> Try except block

In [81]:
# let's create a file
with open('sample.txt','w') as f:
  f.write('hello world')

In [84]:
# try catch demo
try:
  with open('sample1.txt','r') as f:
    print(f.read())
except:
  print('sorry file not found')

sorry file not found


## 3.4 Catching Specific Exceptions

Catching every error with one blanket `except` is often too blunt — you usually want the *right* response per failure. Python lets you stack multiple handlers:

```python
try:
    ...
except FileNotFoundError:
    print('file not found')
except NameError:
    print('variable not defined')
except ZeroDivisionError:
    print(.....)
except Exception as e:
    print(e)
```

Rules that make this work:

- Handlers are checked **in order** against the raised exception class.
- Put the **most specific** exceptions first and the general catch-all (`except Exception as e:`) last, as the safety net.
- `as e` binds the exception object itself, so `print(e)` shows its message (`[Errno 2] No such file or directory: 'sample1.txt'`).

The demo cell raises several candidate errors on purpose: `open()` on a missing file, printing a name, dividing, indexing a list. Only the first matching handler runs — which is why you should keep the order specific → general.

In [96]:
# catching specific exception
try:
  m=5
  f = open('sample1.txt','r')
  print(f.read())
  print(m)
  print(5/2)
  L = [1,2,3]
  L[100]
except FileNotFoundError:
  print('file not found')
except NameError:
  print('variable not defined')
except ZeroDivisionError:
  print("can't divide by 0")
except Exception as e:
  print(e)

[Errno 2] No such file or directory: 'sample1.txt'


## 3.5 The else Clause

A third piece completes the picture: `else`. It appears after the `except` clauses and runs **only if the `try` block completed without any exception**.

```python
try:
    f = open('sample1.txt', 'r')
except FileNotFoundError:
    print('file not found')
else:
    print(f.read())
```

If the file does not exist, the `except` path runs and `else` is skipped; if the file opens happily, `else` proceeds to read it. `else` is valuable because code that should only execute on success is kept visually separate from the risky section — no accidental exception swallowing, clearer intent.

In [98]:
# else
try:
  f = open('sample1.txt','r')
except FileNotFoundError:
  print('file nai mili')
except Exception:
  print('kuch to lafda hai')
else:
  print(f.read())



file nai mili


## 3.6 try / except / else / finally — The Complete Flow

The full pattern is `try → except → else → finally`:

- `try` — the risky code.
- `except` — runs when an exception occurred (one or many, in specificity order).
- `else` — runs only when `try` finished with no exception.
- `finally` — *always* runs, whether an exception happened or not, even if `try` or `except` used `return` or `raise`.

`finally` is the natural home for cleanup: close a file, release a lock, commit or roll back a database connection. Because the language guarantees it runs on every exit path, resources never leak even when the flow is wild.

## 3.7 finally — Guaranteed Cleanup

The demo prints both the failure message (`'ye to print hoga hi'`) and the `finally` message, proving `finally` executes on the error path too. That guarantee is what makes `finally` trustworthy:

- `return` inside `try` → `finally` still runs before the value is handed back.
- `raise` inside `except` → `finally` still runs, then the exception propagates.
- A `while True` loop may be abandoned by an exception → `finally` runs.

Use `finally` for anything that must always happen, and `with` for simpler auto-cleanup of file objects. Together they cover nearly every resource-management need in Python.

In [100]:
# finally
# else
try:
  f = open('sample1.txt','r')
except FileNotFoundError:
  print('file nai mili')
except Exception:
  print('kuch to lafda hai')
else:
  print(f.read())
finally:
  print('ye to print hoga hi')

file nai mili
ye to print hoga hi


# 4. Raising Exceptions Yourself

## 4.1 The raise Keyword

Errors do not have to arrive on their own — you can *throw* them deliberately with `raise`:

```python
raise ZeroDivisionError('aise hi try kar raha hu')
```

Why would you do that? Because detecting an invalid condition and failing fast is often better than continuing with broken state:

- **Input validation** — a negative price, an empty name, a wrong age.
- **Business rules** — not enough balance for a withdrawal.
- **Clarity** — raising a *specific* named exception makes the problem obvious to the caller.

`raise` accepts an exception class and an optional message; `str(e)` recovers the message on the catching side. A raised exception behaves exactly like a natural one: if it is inside a `try`, the matching `except` catches it; otherwise it propagates up the call chain. (Python fans of Java will recognise `raise` ≈ `throw` and `except` ≈ `catch`.) The one anti-pattern to avoid is raising a bare, generic `Exception` everywhere — name your errors precisely.

In [38]:
# raise Exception
# In Python programming, exceptions are raised when errors occur at runtime. 
# We can also manually raise exceptions using the raise keyword.

# We can optionally pass values to the exception to clarify why that exception was raised

## Deep Dive : raise — manual exception

```python
raise ZeroDivisionError('aise hi try kar raha hu')
```

- `raise Exception('message')` — THROW an error yourself
- When: detect an invalid state and fail fast
- Custom business rules: not enough balance, wrong age
- Raise inside try → caught by except
- Common raises: `raise ValueError('invalid input')`
- raise with args → str(e) shows the message
- Can you raise a non-Exception? BaseException exists but it's an anti-pattern
- Re-raise: `raise` inside except (to propagate)
- The generic 'raise Exception' — prefer specific
- Java-style: checked vs unchecked — in Python everything is unchecked


In [106]:
raise ZeroDivisionError('aise hi try kar raha hu')
# Java
# try -> try
# except -> catch
# raise -> throw

ZeroDivisionError: ignored

## 4.2 raise In Business Logic — The Bank Example

The next cell is the classic real-world use of `raise`: an object that refuses to enter an invalid state.

- `Bank.__init__(balance)` stores the starting balance.
- `withdraw(amount)` checks the amount *before* touching the balance:
  - `amount < 0` is nonsense → `raise Exception('amount cannot be -ve')`;
  - `self.balance < amount` is an overdraw → `raise Exception('paise nai hai tere paas')`;
  - otherwise the balance is safely reduced.
- The caller wraps the call in `try/except Exception as e` to catch the message, with an `else` clause that prints the new balance only on success.

Raising here is a form of *validation with a contract*: a method documents what it requires, checks it, and fails loudly rather than silently corrupting the account balance. That is far safer than a method that quietly returns a flag and leaves the data in an impossible state.

In [110]:
class Bank:

  def __init__(self,balance):
    self.balance = balance

  def withdraw(self,amount):
    if amount < 0:
      raise Exception('amount cannot be -ve')
    if self.balance < amount:
      raise Exception('paise nai hai tere paas')
    self.balance = self.balance - amount

obj = Bank(10000)
try:
  obj.withdraw(15000)
except Exception as e:
  print(e)
else:
  print(obj.balance)

paise nai hai tere paas


# 5. Custom Exceptions

## 5.1 Creating Your Own Exception Classes

When the built-in list of exceptions is not expressive enough, define your own. A custom exception is just a class that subclasses `Exception`:

```python
class MyException(Exception):
    def __init__(self, message):
        super().__init__(message)
```

Why bother?

- **Domain vocabulary** — `BankError`, `AuthError`, `DatabaseDown` say more than a forest of generic `Exception`s.
- **Precise catching** — `except MyException` catches exactly that family, nothing else.
- **Cleaner libraries** — users can react to your specific failure types.

The `__init__` passes the message up to the base class so `str(e)` and `print(e)` work as usual. Because your class sits inside the standard hierarchy (`MyException` → `Exception` → `BaseException`), it can be caught either by name or by any ancestor. In the code below the messages are passed to the base this way; the next section shows a tiny slice-down variant where `__init__` prints the text directly.

In [114]:
class MyException(Exception):
  def __init__(self,message):
    print(message)

class Bank:

  def __init__(self,balance):
    self.balance = balance

  def withdraw(self,amount):
    if amount < 0:
      raise MyException('amount cannot be -ve')
    if self.balance < amount:
      raise MyException('paise nai hai tere paas')
    self.balance = self.balance - amount

obj = Bank(10000)
try:
  obj.withdraw(5000)
except MyException as e:
  pass
else:
  print(obj.balance)

5000


## 5.2 A Minimal Custom Exception

A leaner form prints the message straight from `__init__`:

```python
class MyException(Exception):
    def __init__(self, message):
        print(message)
```

Then `raise MyException('amount cannot be -ve')` prints its own message when raised, and the surrounding `try`/`except` can still catch it by name or just `pass`. In the demo, `withdraw(5000)` succeeds (the balance covers it), so no exception fires and the `else` branch prints the updated balance — 5000. This variant is great for compact examples, though storing `self.message` is more robust for code that must inspect the error later.

## 5.3 Why Custom Exceptions Beat Generic Ones

A custom exception carries meaning in its *name*. Compare two handlers:

- `except Exception as e:` tells you almost nothing about *why* — any failure matches.
- `except InsufficientFunds:` tells you the business rule that failed, immediately.

That is why real libraries ship their own exceptions (`requests` raises `requests.exceptions.RequestException`, drivers raise session errors, etc.). When you define `MyException(Exception)`, you claim a place in the hierarchy: anything that catches `Exception` still catches you, but code that only wants *your specific* failures can catch exactly that one class. The comment-heavy cell below hints at this: create custom exceptions, and learn the built-in hierarchy, because the two together give you surgical control over error handling.

In [54]:


# creating custom exceptions
# exception hierarchy in python

## 5.4 Real Scenario — Domain Errors In Action

The `Google`/`SecurityError` demo at the end of the session packs everything together:

- `SecurityError(Exception)` carries a message and an extra `logout()` method — custom behaviour travels with the error.
- `Google.login(email, password, device)` raises `SecurityError` when the login device does not match the registered one — a security-domain violation.
- The caller wraps the login in `try/except SecurityError` and, on failure, does the right thing: logs the user out via `e.logout()`.
- `else` prints the user's name only when login succeeded; `finally` reports that the database connection has been closed.

Read the final output carefully: the security message prints, then `logout`, then the `finally` line — the exception path, the domain response and the guaranteed cleanup, all working together.

## 5.5 Exceptions As Data Integrity Guards

Think of `raise` inside business methods as a defensive wall:

- `withdraw()` validates before mutating — a guard that keeps the balance non-negative.
- A login loop checks the device and aborts with `SecurityError` rather than granting access.
- Methods document their raising conditions, so callers know which `except` clause to prepare.

Compared with returning error codes, exceptions are *explicit*: they cannot be silently ignored, they carry a message, and they integrate with the class hierarchy. Combined with OOP, this style produces clean code where data integrity is enforced at the source instead of being checked four call-sites away. Log a meaningful message when you raise — it doubles as a debugging aid when tracking down *why* a rule fired.

In [57]:
# simple example

# 6. The Exception Hierarchy

## 6.1 One Tree That Explains Everything

All exceptions hang from a single tree:

```
BaseException
 └── Exception
      ├── ArithmeticError
      │    └── ZeroDivisionError
      ├── LookupError
      │    ├── IndexError
      │    └── KeyError
      ├── ValueError
      ├── TypeError
      ├── FileNotFoundError
      └── YourCustomException  (subclass of Exception)
```

What the tree tells you:

- `BaseException` is the root; `Exception` is the parent of everything you will normally raise or catch.
- Catching a **parent** catches its **children**: `except LookupError` handles both `IndexError` and `KeyError`.
- `KeyboardInterrupt` and `SystemExit` are *siblings* of `Exception` under `BaseException` — a bare `except Exception` deliberately does **not** catch them.
- Your custom classes plug into the tree where you put them, inheriting all ancestor behaviour.

## 6.2 Navigating The Hierarchy

Practical takeaways:

- **Order matters**: put child exceptions first, generic parents last. `except (IndexError, KeyError):` groups siblings with a tuple.
- **Inspect with MRO**: `MyException.__mro__` shows your class's full ancestor chain.
- **Catch-all meaning**: `except Exception:` covers user-program errors but not `KeyboardInterrupt` or `SystemExit` — exactly the safety you usually want in an app's outer shell.
- **Docs**: `help('builtins')` or `dir(__builtins__)` lists the complete built-in family when you need the full picture.

Knowing the parents is knowing what your `except` catches — and that single fact prevents the most embarrassing handling bugs: catching the wrong sibling, forgetting a child, or ducking the keyboard interrupt the user expects to stop the program.

In [117]:
class SecurityError(Exception):

  def __init__(self,message):
    print(message)

  def logout(self):
    print('logout')

class Google:

  def __init__(self,name,email,password,device):
    self.name = name
    self.email = email
    self.password = password
    self.device = device

  def login(self,email,password,device):
    if device != self.device:
      raise SecurityError('bhai teri to lag gayi')
    if email == self.email and password == self.password:
      print('welcome')
    else:
      print('login error')



obj = Google('nitish','nitish@gmail.com','1234','android')

try:
  obj.login('nitish@gmail.com','1234','windows')
except SecurityError as e:
  e.logout()
else:
  print(obj.name)
finally:
  print('database connection closed')



bhai teri to lag gayi
logout
database connection closed


## Summary

Session 11 turned errors from enemies into tools. You distinguished syntax errors (compile-time failures fixed by correcting grammar) from exceptions (runtime failures handled in code). The built-in family — `IndexError`, `KeyError`, `TypeError`, `ValueError`, `NameError`, `AttributeError`, and more — names its cause, so reading tracebacks is your first diagnostic skill. The `try`/`except` block wrapped around risky code catches problems gracefully; stacked `except` clauses handle each failure type when ordered specific → general; `else` runs only on success; and `finally` always runs, making it the home for cleanup like closing files and connections. With `raise` you throw exceptions deliberately — the right response to invalid business rules such as an overdraw — and custom exception classes (subclasses of `Exception`) signal domain problems by name while fitting into the one hierarchy rooted at `BaseException`. Fail loudly, recover gracefully, and keep your programs readable.